# Section 1: Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

# Section 2: Load Data

In [2]:
model_df = pd.read_parquet(
    "../data/processed/model_df.parquet"
)

print(model_df.shape)

model_df.head()

(68810, 5)


,customer_unique_id,product_id,product_category_name,price,order_purchase_timestamp
0,861eff4711a542e4b93843c6dd7febb0,a9516a079e37a9c9c36b9b78b10169e8,moveis_escritorio,124.99,2017-05-16 15:05:35
1,290c77bc529b7ac935b93aa66c333dc3,4aa6014eceb682077f9dc4bffebc05b0,utilidades_domesticas,289.00,2018-01-12 20:48:24
2,060e732b5b29e8181a18229c7b0b2b5e,bd07b66896d6f1494f5b86251848ced7,moveis_escritorio,139.94,2018-05-19 16:07:45
3,259dac757896d24d7702b9acbbff3f3c,a5647c44af977b148e0a3a4751a09e2e,moveis_escritorio,149.94,2018-03-13 16:06:38
4,4c93744516667ad3b8f1fb645a3116a4,0be701e03657109a8a4d5168122777fb,esporte_lazer,259.90,2017-09-14 18:14:31


# Section 3: Train/Test Split

In [3]:
split_date = "2018-06-30"

train_df = model_df[
    model_df["order_purchase_timestamp"] <= split_date
].copy()

test_df = model_df[
    model_df["order_purchase_timestamp"] > split_date
].copy()

print(train_df.shape)
print(test_df.shape)

(61419, 5)
(7391, 5)


# Section 4: Customer Favorite Category

In [4]:
customer_category_counts = (
    train_df
    .groupby(
        [
            "customer_unique_id",
            "product_category_name"
        ]
    )
    .size()
    .reset_index(name="purchase_count")
)

customer_category_counts.head()

,customer_unique_id,product_category_name,purchase_count
0,0000366f3b9a7992bf8c76cfdf3221e2,cama_mesa_banho,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,beleza_saude,1
2,0000f46a3911fa3c0805444483337064,papelaria,1
3,00050ab1314c0e55a6ca13cf7181fecf,telefonia,1
4,00053a61a98854899e70ed204dd4bafe,esporte_lazer,1


In [5]:
favorite_category_df = (
    customer_category_counts
    .sort_values(
        "purchase_count",
        ascending=False
    )
    .drop_duplicates(
        subset=["customer_unique_id"]
    )
)

favorite_category_df.head()

,customer_unique_id,product_category_name,purchase_count
14239,4546caea018ad8c692964e3382debd19,beleza_saude,20
40055,c402f431464c72e27330a67f7b94d4fb,informatica_acessorios,20
21599,698e1cf81d01a3d389d96145f7fa6df8,automotivo,20
3122,0f5ac8d5c31de21d2f25e24be15bbffb,moveis_decoracao,18
47980,eae0a83d752b1dd32697e0e7b4221656,moveis_escritorio,15


# Section 5: Category Popularity Table 

In [6]:
category_popularity = (
    train_df
    .groupby(
        [
            "product_category_name",
            "product_id"
        ]
    )
    .size()
    .reset_index(name="purchase_count")
)

In [7]:
category_popularity = (
    category_popularity
    .sort_values(
        [
            "product_category_name",
            "purchase_count"
        ],
        ascending=[True, False]
    )
)

category_popularity.head(20)

,product_category_name,product_id,purchase_count
0,agro_industria_e_comercio,11250b0d4b709fee92441c5f34122aed,21
4,agro_industria_e_comercio,672e757f331900b9deea127a2a7b79fd,17
3,agro_industria_e_comercio,423a6644f0aa529e8828ff1f91003690,15
2,agro_industria_e_comercio,3bebad3cf2c8d1a8d3ce97174643e054,13
7,agro_industria_e_comercio,a0fe1efb855f3e786f0650268cd77f44,12
1,agro_industria_e_comercio,16d096faa27582985f849f08370cf1ed,7
9,agro_industria_e_comercio,b5aebb467d9a92162173cbd234e00d99,7
5,agro_industria_e_comercio,6ff1fc9209c7854704a4f75c9fac41b4,6
10,agro_industria_e_comercio,c183fd5d2abf05873fa6e1014ed9e06c,6
6,agro_industria_e_comercio,980ecbcc15fe174ec1e5757c4d75b1bf,5


# Section 6: Recommendation Function

In [8]:
def recommend_by_category(
    customer_id,
    favorite_category_df,
    category_popularity,
    top_k=10
):
    
    customer_row = favorite_category_df[
        favorite_category_df["customer_unique_id"]
        == customer_id
    ]
    
    if len(customer_row) == 0:
        return pd.DataFrame()
    
    category = customer_row.iloc[0][
        "product_category_name"
    ]
    
    recommendations = (
        category_popularity[
            category_popularity[
                "product_category_name"
            ] == category
        ]
        .head(top_k)
    )
    
    return recommendations

# Section 7: Test One Customer

In [9]:
sample_customer = (
    favorite_category_df[
        "customer_unique_id"
    ]
    .iloc[0]
)

sample_customer

'4546caea018ad8c692964e3382debd19'

In [10]:
recommend_by_category(
    sample_customer,
    favorite_category_df,
    category_popularity,
    top_k=10
)

,product_category_name,product_id,purchase_count
407,beleza_saude,154e7e31ebfa092203795c972e5804a6,276
439,beleza_saude,2b4609f8948be18874494203496bc318,243
553,beleza_saude,7c1bd920dbdf22470b68bde975dd3ccf,226
647,beleza_saude,bb50f2e236e5eea0100680137654686c,154
471,beleza_saude,437c05a395e9e47f9762e677a7068ce7,139
533,beleza_saude,6cdd53843498f92890544667809f1595,136
485,beleza_saude,4c2394abfbac7ff59ec7a420918562fa,123
693,beleza_saude,e0cf79767c5b016251fe139915c59a26,105
463,beleza_saude,3fbc0ef745950c7932d5f2a446189725,95
416,beleza_saude,19c91ef95d509ea33eda93495c4d3481,91


In [11]:
print(favorite_category_df.shape)

print(category_popularity.shape)

recommend_by_category(
    sample_customer,
    favorite_category_df,
    category_popularity
)

(51613, 3)
(4683, 3)


,product_category_name,product_id,purchase_count
407,beleza_saude,154e7e31ebfa092203795c972e5804a6,276
439,beleza_saude,2b4609f8948be18874494203496bc318,243
553,beleza_saude,7c1bd920dbdf22470b68bde975dd3ccf,226
647,beleza_saude,bb50f2e236e5eea0100680137654686c,154
471,beleza_saude,437c05a395e9e47f9762e677a7068ce7,139
533,beleza_saude,6cdd53843498f92890544667809f1595,136
485,beleza_saude,4c2394abfbac7ff59ec7a420918562fa,123
693,beleza_saude,e0cf79767c5b016251fe139915c59a26,105
463,beleza_saude,3fbc0ef745950c7932d5f2a446189725,95
416,beleza_saude,19c91ef95d509ea33eda93495c4d3481,91


# Section 8: Build Ground Truth

In [12]:
test_ground_truth = (
    test_df
    .groupby("customer_unique_id")["product_id"]
    .apply(set)
    .to_dict()
)

len(test_ground_truth)

6502

# Section 9: Precision@10 Evaluation

In [13]:
TOP_K = 10

hits = 0
total_recommendations = 0

evaluation_customers = set(
    test_ground_truth.keys()
).intersection(
    set(
        favorite_category_df[
            "customer_unique_id"
        ]
    )
)

len(evaluation_customers)

99

In [14]:
for customer_id in evaluation_customers:
    
    recommended_products = (
        recommend_by_category(
            customer_id,
            favorite_category_df,
            category_popularity,
            top_k=TOP_K
        )["product_id"]
        .tolist()
    )
    
    actual_products = (
        test_ground_truth[
            customer_id
        ]
    )
    
    customer_hits = len(
        set(recommended_products)
        &
        actual_products
    )
    
    hits += customer_hits
    
    total_recommendations += TOP_K

In [15]:
precision_at_10 = (
    hits / total_recommendations
)

print(
    f"Precision@10: {precision_at_10:.4f}"
)

Precision@10: 0.0081


In [16]:
recommended_products = set()

for customer_id in evaluation_customers:
    
    recs = recommend_by_category(
        customer_id,
        favorite_category_df,
        category_popularity,
        top_k=TOP_K
    )["product_id"]
    
    recommended_products.update(
        recs.tolist()
    )

coverage = (
    len(recommended_products)
    /
    train_df["product_id"].nunique()
)

print(
    f"Coverage: {coverage:.4f}"
)

Coverage: 0.0590


In [17]:
print(precision_at_10)
print(coverage)
print(len(evaluation_customers))

0.00808080808080808
0.05903436643474594
99


In [18]:
train_customers = set(
    train_df["customer_unique_id"]
)

test_customers = set(
    test_df["customer_unique_id"]
)

common_customers = (
    train_customers &
    test_customers
)

print("Train Customers :", len(train_customers))
print("Test Customers  :", len(test_customers))
print("Common Customers:", len(common_customers))

Train Customers : 52277
Test Customers  : 6502
Common Customers: 101


In [19]:
favorite_category_df.to_parquet(
    "../artifacts/customer_favorite_category.parquet",
    index=False
)

In [21]:
category_popularity.to_parquet(
    "../artifacts/category_popularity.parquet",
    index=False
)

In [23]:
import json

model_2_metrics = {
    "precision_at_10": float(precision_at_10),
    "coverage": float(coverage),
    "evaluated_customers": int(len(evaluation_customers))
}

with open(
    "../artifacts/model_2_metrics.json",
    "w"
) as f:
    json.dump(
        model_2_metrics,
        f,
        indent=4
    )